# NB01 — Data EDA: Pair D simple-β empirical validation

**Spec:** `2026-04-27-simple-beta-pair-d-design.md` v1.3.1, sha256 `964c62cca0be1b9070944b5398fe97886c6d07d37ba7121199de8ccc341ef659`.

**Hypothesis:** β_composite = β_6 + β_9 + β_12 > 0 in OLS of `Y_broad_logit` on `log_cop_usd_lag6 + log_cop_usd_lag9 + log_cop_usd_lag12 + intercept`, where Y is the Colombian young-worker (14-28) services-sector employment share (DANE GEIH micro-data, broad CIIU Rev.4 G-T).

**Window:** 2015-01-31 → 2026-02-28 (UTC, month-end), N = 134 (post-CORRECTIONS-α' Option-α').

**Authoring discipline:** trio-checkpoint per memory `feedback_notebook_trio_checkpoint`. Each analytical decision is a `(why-markdown → code-cell → interpretation-markdown)` trio with HALT for orchestrator review. Re-execution of the script-form Phase-2 (committed at `bce431544`) under Option-β chosen 2026-04-28 PM late evening; sha256 round-trip is asserted against the committed JSON artifacts.

---

## §1 Panel load + descriptive stats

Trio 1 — load `panel_combined.parquet` (sha256 `6d7d9e60dad1715ce86e8adb7b3d44ba236d0b063796293b40575994a9363edf`); compute summary stats (N, Y range, X range, missing-data check); fingerprint to `estimates/nb1_panel_fingerprint.json`.

**Trio 1 — why-block (4-part citation block per `feedback_notebook_citation_block`)**

1. **Reference.** Spec [@abrigoSimpleBetaSpec2026] §4 fixes the analysis window 2015-01-31 → 2026-02-28 (UTC, month-end) at N = 134 observations under CORRECTIONS-α' Option-α' — the post-Marco-2018-break narrowing motivated by the DANE 2021 methodology break [@dane2021empalme]. §5.1 constructs `Y_broad` and `Y_narrow` as the Colombian young-worker (14–28) services-sector employment share over CIIU Rev.4 sections G–T (broad) and J+M+N (BPO-narrow); both are pinned to be interior to (0, 1) so the logit transform is licensed without ad-hoc clipping. §5.2 constructs `log_cop_usd_lag{6,9,12}` as natural-log monthly end-of-period COP/USD spot from Banco de la República TRM [@banrepCOPUSD], lagged by the BPO offshoring-contracting-cycle horizon literature-grounded in the Baumol-cost-disease → US-Colombia service-wage arbitrage chain [@mendietaMunoz2017colombia; @rodrik2016premature; @baumolBowen1966performing]. The spec sha256 is pinned at `964c62cca0be1b9070944b5398fe97886c6d07d37ba7121199de8ccc341ef659` (env.SPEC_SHA256).

2. **Why used (the EDA gate).** Before any estimation in NB02 or any sensitivity in NB03, this trio verifies four pre-registered invariants on the panel: (a) the time index matches the spec window edge-to-edge (no off-by-one); (b) `len(df) == 134`, the post-lag-12 sample size that drove the N_MIN feasibility argument; (c) both Y series lie strictly inside (0, 1) so `Y_logit = log(Y / (1 − Y))` is finite at every row; (d) the three lag columns and the two Y columns have zero missing rows. A failure on any one is HALT-class — it would mean the Phase-1 panel-builder script (`scripts/task_1_3_panel_align.py`) drifted from the spec it was written against, and the byte-deterministic re-execution path under Option-β cannot proceed.

3. **Relevance to results.** A pass on this trio is the green light for NB02 §1 (which assumes the panel is canonical) and NB03 §1 (which re-loads the same parquet for the R1–R4 robustness re-estimations). The fingerprint JSON written here becomes the upstream sha256-pinned input for both downstream notebooks; any future re-run that fails the round-trip will surface immediately at NB02 load, rather than producing silently stale estimates.

4. **Connection to simulator (Stage-2 M-sketch).** The empirical ranges of `Y_broad` and the three lagged log COP/USD columns characterize the support of the (Y, X) joint distribution that an ideal-scenario Panoptic perpetual would settle against. Specifically, the observed Y-range bounds the strike geometry of the would-be put/call structure on the Colombian-structural-transformation index, and the X-range bounds the underlying-price grid the Panoptic position is parameterized over. The descriptives this trio emits feed directly into the Stage-2 M-sketch's parameter selection — they are not throwaway sanity checks.

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

import env

# ── Determinism ────────────────────────────────────────────────────────────
env.pin_seed(env.SEED)

# ── sha256 round-trip on the Phase-1 panel ─────────────────────────────────
panel_bytes = env.PANEL_PARQUET.read_bytes()
panel_sha256_observed = hashlib.sha256(panel_bytes).hexdigest()
assert panel_sha256_observed == env.PANEL_SHA256, (
    f"Panel parquet sha256 drift: observed {panel_sha256_observed!r}, "
    f"pinned {env.PANEL_SHA256!r}. HALT — re-running Phase-1 is required "
    f"before any EDA can proceed."
)

# ── Load panel ─────────────────────────────────────────────────────────────
df = pd.read_parquet(env.PANEL_PARQUET)

# ── Invariant: N = 134 ─────────────────────────────────────────────────────
assert len(df) == env.N_EXPECTED, (
    f"N invariant violated: len(df)={len(df)}, expected {env.N_EXPECTED}. "
    f"HALT — Phase-1 panel-builder drifted from spec §4."
)

# ── Invariant: window edges match spec §4 ──────────────────────────────────
ts = pd.to_datetime(df["timestamp_utc"], utc=True).sort_values().reset_index(drop=True)
window_start_observed = ts.iloc[0].strftime("%Y-%m-%d")
window_end_observed = ts.iloc[-1].strftime("%Y-%m-%d")
assert window_start_observed == env.WINDOW_START, (
    f"Window-start drift: observed {window_start_observed!r}, "
    f"pinned env.WINDOW_START={env.WINDOW_START!r}."
)
assert window_end_observed == env.WINDOW_END, (
    f"Window-end drift: observed {window_end_observed!r}, "
    f"pinned env.WINDOW_END={env.WINDOW_END!r}."
)

# ── Required columns present ───────────────────────────────────────────────
required_cols = [
    "timestamp_utc",
    "Y_broad_raw",
    "Y_broad_logit",
    "Y_narrow_raw",
    "Y_narrow_logit",
    "log_cop_usd_lag6",
    "log_cop_usd_lag9",
    "log_cop_usd_lag12",
]
missing_cols = [c for c in required_cols if c not in df.columns]
assert not missing_cols, f"Missing required columns: {missing_cols}"

# ── Y range checks (spec §5.1 pre-registered bounds) ───────────────────────
y_broad_min, y_broad_max = float(df["Y_broad_raw"].min()), float(df["Y_broad_raw"].max())
y_narrow_min, y_narrow_max = float(df["Y_narrow_raw"].min()), float(df["Y_narrow_raw"].max())

assert 0.55 <= y_broad_min and y_broad_max <= 0.75, (
    f"Y_broad outside spec §5.1 pre-registered range [0.55, 0.75]: "
    f"observed [{y_broad_min:.6f}, {y_broad_max:.6f}]."
)
assert 0.07 <= y_narrow_min and y_narrow_max <= 0.12, (
    f"Y_narrow outside spec §5.1 pre-registered range [0.07, 0.12]: "
    f"observed [{y_narrow_min:.6f}, {y_narrow_max:.6f}]."
)

# ── Logit validity: strict interior (0, 1) ────────────────────────────────
assert (df["Y_broad_raw"] > 0).all() and (df["Y_broad_raw"] < 1).all(), (
    "Y_broad_raw escaped (0, 1); logit transform would be non-finite."
)
assert (df["Y_narrow_raw"] > 0).all() and (df["Y_narrow_raw"] < 1).all(), (
    "Y_narrow_raw escaped (0, 1); logit transform would be non-finite."
)

# ── X range descriptives (log COP/USD lag-6/9/12) ──────────────────────────
x_lag6_min, x_lag6_max = float(df["log_cop_usd_lag6"].min()), float(df["log_cop_usd_lag6"].max())
x_lag9_min, x_lag9_max = float(df["log_cop_usd_lag9"].min()), float(df["log_cop_usd_lag9"].max())
x_lag12_min, x_lag12_max = float(df["log_cop_usd_lag12"].min()), float(df["log_cop_usd_lag12"].max())

# ── Missing-data check (zero NaN tolerance on required cols) ───────────────
n_missing = int(df[required_cols].isna().sum().sum())
assert n_missing == 0, (
    f"Missing data in required columns: {n_missing} cell(s). "
    f"Phase-1 panel-builder is supposed to dropna; HALT."
)

# ── Fingerprint dict ───────────────────────────────────────────────────────
fingerprint = {
    "n": int(len(df)),
    "panel_sha256": env.PANEL_SHA256,
    "spec_sha256": env.SPEC_SHA256,
    "window_start": env.WINDOW_START,
    "window_end": env.WINDOW_END,
    "y_broad_range": [y_broad_min, y_broad_max],
    "y_narrow_range": [y_narrow_min, y_narrow_max],
    "x_lag6_range": [x_lag6_min, x_lag6_max],
    "x_lag9_range": [x_lag9_min, x_lag9_max],
    "x_lag12_range": [x_lag12_min, x_lag12_max],
    "missing_cells_required_cols": n_missing,
    "required_cols": required_cols,
}

# ── Persist fingerprint for NB02 §1 + NB03 §1 handoff ──────────────────────
env.PANEL_FINGERPRINT_PATH.parent.mkdir(parents=True, exist_ok=True)
env.PANEL_FINGERPRINT_PATH.write_text(
    json.dumps(fingerprint, indent=2, sort_keys=True) + "\n"
)

print("── NB01 fingerprint ──")
print(json.dumps(fingerprint, indent=2, sort_keys=True))
print(f"\nWritten to: {env.PANEL_FINGERPRINT_PATH}")

# ── Final cell expression: full descriptive summary ────────────────────────
df.describe()

── NB01 fingerprint ──
{
  "missing_cells_required_cols": 0,
  "n": 134,
  "panel_sha256": "6d7d9e60dad1715ce86e8adb7b3d44ba236d0b063796293b40575994a9363edf",
  "required_cols": [
    "timestamp_utc",
    "Y_broad_raw",
    "Y_broad_logit",
    "Y_narrow_raw",
    "Y_narrow_logit",
    "log_cop_usd_lag6",
    "log_cop_usd_lag9",
    "log_cop_usd_lag12"
  ],
  "spec_sha256": "964c62cca0be1b9070944b5398fe97886c6d07d37ba7121199de8ccc341ef659",
  "window_end": "2026-02-28",
  "window_start": "2015-01-31",
  "x_lag12_range": [
    7.534992331515036,
    8.480408867853969
  ],
  "x_lag6_range": [
    7.534992331515036,
    8.480408867853969
  ],
  "x_lag9_range": [
    7.534992331515036,
    8.480408867853969
  ],
  "y_broad_range": [
    0.6022439073932842,
    0.6840826007872685
  ],
  "y_narrow_range": [
    0.0726495265130294,
    0.11578908547588847
  ]
}

Written to: /home/jmsbpp/apps/ThetaSwap/thetaSwap-core-dev/.worktree/ranFromAngstrom/contracts/.scratch/simple-beta-pair-d/notebooks

,Y_broad_logit,Y_broad_raw,Y_narrow_logit,Y_narrow_raw,log_cop_usd_lag6,log_cop_usd_lag9,log_cop_usd_lag12,n_young_employed,n_young_in_services_broad,n_young_in_services_narrow
count,134.000000,134.000000,134.000000,134.000000,134.000000,134.000000,134.000000,134.000000,134.000000,134.000000
mean,0.595951,0.644582,-2.284813,0.093030,8.139529,8.122436,8.106035,7007.723881,5142.888060,637.582090
std,0.067188,0.015386,0.137308,0.011645,0.199331,0.215653,0.226321,909.654073,652.493571,70.912771
min,0.414824,0.602244,-2.546685,0.072650,7.534992,7.534992,7.534992,4464.000000,3368.000000,443.000000
25%,0.557819,0.635948,-2.398438,0.083292,7.998043,7.986089,7.984302,6267.500000,4624.750000,589.000000
50%,0.594906,0.644490,-2.297493,0.091331,8.158263,8.137209,8.111271,7044.500000,5130.500000,645.000000
75%,0.638724,0.654465,-2.164620,0.102973,8.289820,8.288299,8.275140,7738.250000,5683.500000,691.000000
max,0.772598,0.684083,-2.032925,0.115789,8.480409,8.480409,8.480409,8974.000000,6556.000000,791.000000


**Trio 1 — interpretation**

- **N invariant.** `len(df) == 134` confirmed. Spec §4 window edges asserted edge-to-edge: 2015-01-31 → 2026-02-28 (UTC, month-end). This is the post-CORRECTIONS-α' Option-α' window agreed at HEAD `21beffade` after the Empalme schema-stability second HALT (`PairDEmpalmeSchemaContradictsHarmonizationPin` typed exception); pre-α' work on 2008→2010 / 2010→2014 windows is superseded.

- **Y_broad observed range.** Strictly interior to the spec §5.1 pre-registered band [0.55, 0.75]; interior-to-(0, 1) means the logit transform `log(Y / (1 − Y))` is finite at every row, so the spec §5.1 logit-Y primary specification is licensed without ad-hoc clipping or epsilon-padding. The exact (min, max) is emitted in the printed fingerprint above and persisted to `estimates/nb1_panel_fingerprint.json`.

- **Y_narrow observed range.** BPO-narrow (CIIU Rev.4 J+M+N) subset; observed range strictly interior to spec §5.1 pre-registered band [0.07, 0.12]. Same logit-validity argument applies. Y_narrow is the pre-registered sensitivity Y (spec §5.1, secondary), used in NB03 R2.

- **X lag-6/9/12 ranges (log COP/USD).** All three lag columns finite, no infinities, no NaN; ranges reflect the 2008-mid → 2025-late COP/USD trajectory (the lag-12 column starts 12 months before the panel window because the panel-builder applies the lag and then dropna). The X grid this defines is the underlying-price support a Stage-2 Panoptic-position M-sketch would parameterize over.

- **Missing-data status.** Zero missing cells across all eight required columns. Phase-1 panel-builder (`scripts/task_1_3_panel_align.py`) honors its `dropna` contract.

- **Fingerprint persistence.** Written to `contracts/.scratch/simple-beta-pair-d/notebooks/estimates/nb1_panel_fingerprint.json`. NB02 §1 and NB03 §1 will sha256-load this JSON to confirm they are operating on the identical panel state EDA validated.

- **Reproducibility chain.** Panel sha256 `6d7d9e60dad1715ce86e8adb7b3d44ba236d0b063796293b40575994a9363edf` matched env.PANEL_SHA256; spec sha256 `964c62cca0be1b9070944b5398fe97886c6d07d37ba7121199de8ccc341ef659` matched env.SPEC_SHA256. The Phase-2 byte-deterministic re-execution path under Option-β is intact.

**Trio 1 PASS. NB02 may proceed.**